In [1]:
library(hise)
library(dplyr)
library(msigdbr)
library(purrr)
library(jsonlite)

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘msigdbr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘jsonlite’ was built under R version 4.4.3”

Attaching package: ‘jsonlite’


The following object is masked from ‘package:purrr’:

    flatten




In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}

## Download Reactome gene sets

We'll obtain the Reactome pathways and relationships provided by reactome.org

In [65]:
# Pathway identifiers, names, and species
download.file(
    "https://download.reactome.org/87/ReactomePathways.txt", 
    "ReactomePathways.txt"
)
# Pathway gene sets
download.file(
    "https://download.reactome.org/87/ReactomePathways.gmt.zip", 
    "ReactomePathways.gmt.zip"
)
system("unzip ReactomePathways.gmt.zip")
# Pathway relationships
download.file(
    "https://download.reactome.org/87/ReactomePathwaysRelation.txt",
    "ReactomePathwaysRelation.txt"
)

## Read, filter, and structure gene sets and relationships

### Read and convert .gmt

The gene set GMT file contains one gene set per line, with the set name and id followed by the list of genes.

We'll read these lines, split on tabs, and then use the split data to build a tibble in which each row has a name, id, and gene list.

In [66]:
sets <- readLines("ReactomePathways.gmt")
sets <- strsplit(sets, split = "\t")

In [67]:
sets <- map_dfr(
    sets,
    function(v) {
        tibble(
            name = map_chr(sets, 1),
            id = map_chr(sets, 2),
            genes = lapply(sets, function(x) {x[-c(1,2)]})
        )
})

In [68]:
sets <- sets %>%
  select(id, genes) %>%
  unique()

In [69]:
nrow(sets)

[1] 2656

### Read set IDs and filter for human pathways

Next, we'll read the ReactomePathways file, and use the 3rd column to filter for gene sets from *Homo sapiens*.

In [70]:
pw <- readLines("ReactomePathways.txt")
pw <- strsplit(pw, split = "\t")

In [71]:
pw <- map(pw, as.list)
pw <- map(pw, function(l) { names(l) <- c("id", "name", "species"); l })
pw <- map_dfr(pw, as.data.frame)

In [72]:
head(pw)

,id,name,species
,<chr>,<chr>,<chr>
1,R-BTA-73843,5-Phosphoribose 1-diphosphate biosynthesis,Bos taurus
2,R-BTA-1971475,A tetrasaccharide linker sequence is required for GAG synthesis,Bos taurus
3,R-BTA-1369062,ABC transporters in lipid homeostasis,Bos taurus
4,R-BTA-382556,ABC-family proteins mediated transport,Bos taurus
5,R-BTA-9033807,ABO blood group biosynthesis,Bos taurus
6,R-BTA-418592,ADP signalling through P2Y purinoceptor 1,Bos taurus


In [73]:
pw <- pw %>%
  filter(species == "Homo sapiens") %>%
  select(id, name)

In [74]:
nrow(pw)

[1] 2673

### Load relationships

The last piece required is the relationships between pathways, which we'll structure as a data.frame.

In [75]:
links <- read.table("ReactomePathwaysRelation.txt", sep = "\t")
names(links) <- c("from", "to")
links <- links %>%
  filter(from %in% pw$id)

In [76]:
nrow(links)

[1] 2691

### Identify root nodes

To find the major pathway root nodes and links, we'll select pathways that link *to* other nodes, but don't have links *from* any parent nodes.

In [77]:
root <- pw %>%
  filter(id %in% links$from & !id %in% links$to) %>%
  left_join(sets)
names(root) <- c("root_id", "root_name", "root_genes")
root_links <- links %>%
  filter(from %in% root$root_id)

Joining with `by = join_by(id)`


In [78]:
root$root_name

[1] "Autophagy"                           
 [2] "Cell Cycle"                          
 [3] "Cell-Cell communication"             
 [4] "Cellular responses to stimuli"       
 [5] "Chromatin organization"              
 [6] "Circadian Clock"                     
 [7] "DNA Repair"                          
 [8] "DNA Replication"                     
 [9] "Developmental Biology"               
[10] "Digestion and absorption"            
[11] "Disease"                             
[12] "Drug ADME"                           
[13] "Extracellular matrix organization"   
[14] "Gene expression (Transcription)"     
[15] "Hemostasis"                          
[16] "Immune System"                       
[17] "Metabolism"                          
[18] "Metabolism of RNA"                   
[19] "Metabolism of proteins"              
[20] "Muscle contraction"                  
[21] "Neuronal System"                     
[22] "Organelle biogenesis and maintenance"
[23] "Programmed Cell Death"               
[24] "Protein localization"                
[25] "Reproduction"                        
[26] "Sensory Perception"                  
[27] "Signal Transduction"                 
[28] "Transport of small molecules"        
[29] "Vesicle-mediated transport"

In [79]:
nrow(root)

[1] 29

In [80]:
length(unique(root$root_id))

[1] 29

## Identify sub-pathways

For our analysis, we'll use sub-pathways that are up to 4 levels below the Root nodes. For each level, we assemble the gene set and keep track of parent gene sets.

## Level 1

Just below the top nodes

In [81]:
root_split <- split(root_links, root_links$from)
l1 <- map2_dfr(
    root_split, names(root_split),
    function(link_df, parent_id) {
        parent_pw <- root %>%
          filter(root_id == parent_id)
        l1_pw <- pw %>%
          filter(id %in% link_df$to)
        names(l1_pw) <- c("l1_id", "l1_name")
        l1_pw <- l1_pw %>%
          mutate(root_id = parent_id) %>%
          left_join(root, by = "root_id")
        l1_pw
    }
)

In [82]:
l1 <- l1 %>%
  left_join(sets, by = c("l1_id" = "id"))

In [83]:
names(l1)[length(l1)] <- "l1_genes"

In [84]:
l1 <- l1 %>%
  filter(map_int(l1_genes, length) > 10)

Remove double parentage

In [85]:
n_links <- nrow(l1)
n_links

[1] 157

In [86]:
n_targets <- length(unique(l1$l1_id))
n_targets

[1] 156

In [87]:
l1 <- l1 %>%
  group_by(l1_id) %>%
  slice(1) %>%
  ungroup()

In [88]:
l1_links <- links %>%
  filter(from %in% l1$l1_id)

Filter available pathways to prevent double nesting at lower levels

In [89]:
filtered_pw <- pw %>%
  filter(!id %in% l1$l1_id)

## Level 2
Children of Level 1 nodes

In [90]:
l1_split <- split(l1_links, l1_links$from)
l2 <- map2_dfr(
    l1_split, names(l1_split),
    function(link_df, parent_id) {
        parent_pw <- l1 %>%
          filter(l1_id == parent_id)
        l2_pw <- filtered_pw %>%
          filter(id %in% link_df$to)
        names(l2_pw) <- c("l2_id", "l2_name")
        l2_pw <- l2_pw %>%
          mutate(l1_id = parent_id) %>%
          left_join(parent_pw, by = "l1_id")
        l2_pw
    }
)

In [91]:
l2 <- l2 %>%
  left_join(sets, by = c("l2_id" = "id"))

In [92]:
names(l2)[length(l2)] <- "l2_genes"

In [93]:
l2 <- l2 %>%
  filter(map_int(l2_genes, length) > 10)

In [94]:
n_links <- nrow(l2)
n_links

[1] 365

In [95]:
n_targets <- length(unique(l2$l2_id))
n_targets

[1] 365

In [96]:
l2 <- l2 %>%
  group_by(l2_id) %>%
  slice(1) %>%
  ungroup()

In [97]:
l2_links <- links %>%
  filter(from %in% l2$l2_id)

In [98]:
filtered_pw <- filtered_pw %>%
  filter(!id %in% l2$l2_id)

## Level 3
Children of Level 2 nodes

In [99]:
l2_split <- split(l2_links, l2_links$from)
l3 <- map2_dfr(
    l2_split, names(l2_split),
    function(link_df, parent_id) {
        parent_pw <- l2 %>%
          filter(l2_id == parent_id)
        l3_pw <- filtered_pw %>%
          filter(id %in% link_df$to)
        names(l3_pw) <- c("l3_id", "l3_name")
        l3_pw <- l3_pw %>%
          mutate(l2_id = parent_id) %>%
          left_join(parent_pw, by = "l2_id")
        l3_pw
    }
)

In [100]:
l3 <- l3 %>%
  left_join(sets, by = c("l3_id" = "id"))
names(l3)[length(l3)] <- "l3_genes"

In [101]:
l3 <- l3 %>%
  filter(map_int(l3_genes, length) > 10)

In [102]:
n_links <- nrow(l3)
n_links

[1] 466

In [103]:
n_targets <- length(unique(l3$l3_id))
n_targets

[1] 464

In [104]:
l3 <- l3 %>%
  group_by(l3_id) %>%
  slice(1) %>%
  ungroup()

In [105]:
l3_links <- links %>%
  filter(from %in% l3$l3_id)

In [106]:
filtered_pw <- filtered_pw %>%
  filter(!id %in% l3$l3_id)

## Level 4
Children of Level 3 nodes

In [107]:
l3_split <- split(l3_links, l3_links$from)
l4 <- map2_dfr(
    l3_split, names(l3_split),
    function(link_df, parent_id) {
        parent_pw <- l3 %>%
          filter(l3_id == parent_id)
        l4_pw <- filtered_pw %>%
          filter(id %in% link_df$to)
        names(l4_pw) <- c("l4_id", "l4_name")
        l4_pw <- l4_pw %>%
          mutate(l3_id = parent_id) %>%
          left_join(parent_pw, by = "l3_id")
        l4_pw
    }
)

In [108]:
l4 <- l4 %>%
  left_join(sets, by = c("l4_id" = "id"))
names(l4)[length(l4)] <- "l4_genes"

In [109]:
l4 <- l4 %>%
  filter(map_int(l4_genes, length) > 10)

In [110]:
n_links <- nrow(l4)
n_links

[1] 377

In [111]:
n_targets <- length(unique(l4$l4_id))
n_targets

[1] 371

In [112]:
l4 <- l4 %>%
  group_by(l4_id) %>%
  slice(1) %>%
  ungroup()

In [113]:
nrow(l4)

[1] 371

## Assemble and output gene sets

Now that we've built out the gene sets from each level, we'll assemble these all in a list of gene sets that can be used for GSEA analysis.

We'll save the list of gene sets along with the information from each level, which will be used later for visualization of the GSEA results.

In [114]:
set_list <- c(
    root$genes,
    l1$l1_genes,
    l2$l2_genes,
    l3$l3_genes,
    l4$l4_genes
)

names(set_list) <- c(
    root$id,
    l1$l1_id,
    l2$l2_id,
    l3$l3_id,
    l4$l4_id
)

In [115]:
reactome_list <- map(
    set_list,
    function(l) {
        list(
            "collection" = "Reactome",
            "geneSymbols" = l
        )
    })
names(reactome_list) <- pw$name[match(names(reactome_list), pw$id)]

In [116]:
names(reactome_list)[1]

[1] "Apoptosis"

In [117]:
head(names(reactome_list))

[1] "Apoptosis"                            
[2] "Transmission across Chemical Synapses"
[3] "Signaling by NODAL"                   
[4] "Fertilization"                        
[5] "Mitochondrial protein import"         
[6] "Cytokine Signaling in Immune system"

## Assemble Hallmark list in same structure

In [118]:
hallmark <- msigdbr(species = "human", collection = "H")

In [119]:
hallmark_list <- split(hallmark, hallmark$gs_name)
hallmark_list <- map(hallmark_list, "gene_symbol")

We'll also need a data.frame with the gene sets for our output files. We'll also include labels for display that are specified in `common/gene_sets/hallmark_names.csv`.

In [120]:
hallmark_names <- read.csv("../common/gene_sets/hallmark_names.csv")

In [121]:
hallmark_df <- data.frame(
    pathway = names(hallmark_list),
    n_pathway_genes = map_int(hallmark_list, length),
    pathway_genes = map_chr(hallmark_list, paste, collapse = ";")
)
hallmark_df <- hallmark_df %>%
  left_join(hallmark_names)

Joining with `by = join_by(pathway)`


In [122]:
hallmark_list <- map(
    hallmark_list,
    function(l) {
        list(
            "collection" = "Hallmark",
            "geneSymbols" = l
        )
    })
names(hallmark_list) <- hallmark_df$pathway_label[match(names(hallmark_list), hallmark_df$pathway)]

## Combine lists

In [123]:
gsea_list <- c(hallmark_list, reactome_list)

In [124]:
length(gsea_list)

[1] 1406

## Save to JSON

In [126]:
out_json <- paste0("output/hallmark_reactome-v87_gene_sets_", Sys.Date(), ".json")
write_json(
    gsea_list, 
    path = out_json, 
    pretty = TRUE, 
    auto_unbox = TRUE
)

## Save file to HISE for downstream use

Because we don't use any input files from HISE, this will be stored via watchfolder.

In [64]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/libopenblasp-r0.3.32.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] jsonlite_2.0.0 purrr_1.2.1    msigdbr_26.1.0 dplyr_1.2.1    hise_2.16.0   

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.7.2       cli_3.6.5         rlang_1.2.0      
 [5] generics_0.1.4    textshaping_0.4.0 assertthat_0.2.1  glue_1.8.0       
 [9] htmltools_0.5.9